In [0]:
from pyspark.sql.functions import current_timestamp, col

# Leitura do arquivo XML usando o conector spark-xml
df = (
    spark.read
        .format("xml")                               # Define o leitor para arquivos XML
        .option("rowTag", "item")                    # Cada <item> do XML vira uma linha
        .load("/Volumes/techdados/bronze/vol_landing/order_items.xml")  # Caminho do arquivo
        .withColumn(
            "_source_file",                          # Coluna técnica de rastreabilidade
            col("_metadata.file_path")               # Caminho do arquivo de origem
        )
        .withColumn(
            "_ingestion_timestamp",                  # Coluna técnica de auditoria
            current_timestamp()                      # Timestamp da ingestão
        )
)

# Escrita do DataFrame como tabela Delta na camada Bronze
(
    df.write
        .format("delta")                             # Formato Delta Lake
        .mode("overwrite")                           # Sobrescreve a tabela se existir
        .option("overwriteSchema", "true")           # Permite atualizar o schema
        .saveAsTable("techdados.bronze.order_items")           # Nome da tabela no metastore
)

In [0]:
%sql
CREATE OR REPLACE TABLE techdados.bronze.order_items
USING DELTA
AS
SELECT
    *,
    _metadata.file_path AS _source_file,        -- Caminho do arquivo de origem
    current_timestamp() AS _ingestion_timestamp -- Timestamp da ingestão
FROM read_files(
    '/Volumes/techdados/bronze/vol_landing/order_items.xml',
    format => 'xml',                            -- Formato XML
    rowTag => 'item'                            -- Cada <item> representa uma linha
);

In [0]:
%sql
select * from techdados.bronze.order_items